# 第32章：绩效与推断

先用六期收益核对统计与复利，再计算PSR、DSR、8×3小矩阵的全部六种CSCV划分，并观察200个零技能策略中的冠军。这里不执行历史924划分。CSCV秩≤(N+1)/2与strict秩<N/2是不同阈值。French分支跳过；默认只创建RUN_DIR，不写结果文件。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。运行步骤、数据要求和预期输出见本指南。

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch32_performance_statistics_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. 描述性绩效统计量手算验证（6 期小样本）

In [ ]:
import numpy as np
import pandas as pd
from quant_course.inference import (
    wealth_index,
    cagr,
    max_drawdown,
    calmar_ratio,
    annualized_volatility,
    per_period_sharpe,
    sharpe_ratio,
    sortino_ratio,
    hit_rate,
    profit_factor,
    tracking_error,
    information_ratio,
)

# 6 期教学收益序列（月度频率 A=12, 跨度 0.5 年）
r = np.array([0.04, -0.02, 0.03, -0.01, 0.05, -0.03])
rf = np.full(6, 0.002)
rb = np.array([0.02, -0.01, 0.01, 0.00, 0.03, -0.01])
ann_periods = 12.0
elapsed_years = 0.5

w = wealth_index(r, initial_capital=1000.0)
cagr_val = cagr(1000.0, w[-1], elapsed_years)
mdd = max_drawdown(w)
calm = calmar_ratio(cagr_val, mdd)
vol_ann = annualized_volatility(r, ann_periods)
sr_per = per_period_sharpe(r, rf)
sr_ann = sharpe_ratio(r, rf, ann_periods)
sort_ann = sortino_ratio(r, target_return=0.0, periods_per_year=ann_periods)
hit = hit_rate(r)
pf = profit_factor(r)
te = tracking_error(r, rb, ann_periods)
ir = information_ratio(r, rb, ann_periods)

print(f"期末财富 W_T:        {w[-1]:.6f}")
print(f"年化复合增长率 CAGR: {cagr_val:.4%}")
print(f"最大回撤 MaxDD:      {mdd:.4%}")
print(f"Calmar 比率:         {calm:.4f}")
print(f"年化波动率:          {vol_ann:.4%}")
print(f"逐期 Sharpe:         {sr_per:.4f}")
print(f"年化 Sharpe:         {sr_ann:.4f}")
print(f"年化 Sortino (tau=0): {sort_ann:.4f}")
print(f"胜率 Hit Rate:       {hit:.2%}")
print(f"盈亏比 Profit Factor:{pf:.4f}")
print(f"跟踪误差 TE:         {te:.4%}")
print(f"信息比率 IR:         {ir:.4f}")

## 2. 算术平均 vs 几何平均：波动率拖累效应（Volatility Drag）

In [ ]:
# 极端两期对称波动：第一期 +50%，第二期 -50%
r_two = np.array([0.50, -0.50])
w_two = wealth_index(r_two, initial_capital=1.0)

arith_mean = float(r_two.mean())
geom_mean = float(np.sqrt(1.5 * 0.5) - 1.0)

print(f"初始本金:             1.0000")
print(f"第一期后 (+50%):      {w_two[1]:.4f}")
print(f"第二期后 (-50%):      {w_two[2]:.4f}")
print(f"算术平均收益率:       {arith_mean:.4%}")
print(f"几何平均收益率:       {geom_mean:.4%}")
print(f"财富损失比例:         {(1.0 - w_two[2]):.4%}")

## 3. 概率夏普比率（PSR）：非正态与样本量复核（Bailey & López de Prado 2012a）

In [ ]:
from quant_course.inference import probabilistic_sharpe_ratio

# 复现论文案例：月度收益 n=24, 月度 SR=0.458 (年化 ~1.585)
sr_hat = 0.458
n_24 = 24

# 1. 假设正态分布 (偏度 0, 峰度 3)
psr_norm = probabilistic_sharpe_ratio(sr_hat, benchmark_sr=0.0, n_obs=n_24, skew=0.0, kurt=3.0)

# 2. 考虑对冲基金非正态性 (负偏 -2.448, 厚尾 10.164)
psr_non_norm = probabilistic_sharpe_ratio(sr_hat, benchmark_sr=0.0, n_obs=n_24, skew=-2.448, kurt=10.164)

# 3. 将样本量扩充至 3 年 (n=36)
psr_n36 = probabilistic_sharpe_ratio(sr_hat, benchmark_sr=0.0, n_obs=36, skew=-2.448, kurt=10.164)

print(f"正态假设下 PSR(0) (n=24):              {psr_norm:.4f} (论文: 0.982)")
print(f"非正态修正下 PSR(0) (n=24):            {psr_non_norm:.4f} (同一输入的示例值)")
print(f"样本改为 36 个月后 PSR(0) (n=36):      {psr_n36:.4f} (固定参数敏感度示例)")
print("解释：在 IID、有限四阶矩与渐近正态近似条件下，PSR 是对 H0: SR <= SR* 的单侧近似检验量；原假设边界 SR=SR* 处的单侧近似 p 值为 1 - PSR，不是后验概率。")
print("95% 是可预先选择的决策门槛；本例的 36 个月不是普遍最低样本量。")

## 4. 紧缩夏普比率（DSR）：多重试验与期望最大值门槛（Bailey & López de Prado 2014）

In [ ]:
from quant_course.inference import deflated_sharpe_ratio, implied_independent_trials
from quant_course.research_protocol import expected_max_standard_normal

# 复现 DSR 论文数值案例：日频 T=1250, 年化 SR=2.5 (日频 2.5 / sqrt(250) = 0.1581)
t_obs = 1250
ann_factor = 250.0
sr_daily = 2.5 / np.sqrt(ann_factor)
var_sr_daily = 0.5 / ann_factor  # 年化方差 0.5 -> 日频 0.5/250 = 0.002
skew = -3.0
kurt = 10.0

# 1. N = 100 次独立试验
dsr_100, sr0_100 = deflated_sharpe_ratio(sr_daily, var_sr_daily, 100, t_obs, skew=skew, kurt=kurt)

# 2. N = 46 次独立试验
dsr_46, sr0_46 = deflated_sharpe_ratio(sr_daily, var_sr_daily, 46, t_obs, skew=skew, kurt=kurt)

# 3. 正态分布 (0, 3) 且 N = 88 次试验
dsr_88_norm, sr0_88_norm = deflated_sharpe_ratio(sr_daily, var_sr_daily, 88, t_obs, skew=0.0, kurt=3.0)

print(f"N=100 试验: SR0={sr0_100:.4f} (年化 {sr0_100*np.sqrt(ann_factor):.4f}), DSR={dsr_100:.4f} (论文: 0.9004)")
print(f"N=46  试验: SR0={sr0_46:.4f}  (年化 {sr0_46*np.sqrt(ann_factor):.4f}), DSR={dsr_46:.4f}  (论文: 0.9505)")
print(f"N=88  正态: SR0={sr0_88_norm:.4f} (年化 {sr0_88_norm*np.sqrt(ann_factor):.4f}), DSR={dsr_88_norm:.4f} (论文: 0.9505)")

# 4. 试验相关性插值启发式 N_hat = rho + (1 - rho) * M
m_total = 100
for rho in [0.0, 0.2, 0.5, 0.8, 1.0]:
    n_hat = implied_independent_trials(m_total, rho)
    print(f"  平均相关 rho={rho:.1f} -> 隐含独立试验数 N_hat={n_hat:.1f}")

## 5. 组合对称交叉验证（CSCV）与 PBO 手算例（8 样本 x 3 策略）

In [ ]:
from quant_course.inference import pbo_cscv

rows = [
    [0.02, 0.01, -0.01],
    [0.01, 0.00, 0.02],
    [-0.01, 0.02, 0.00],
    [0.03, -0.01, 0.01],
    [0.00, 0.01, 0.02],
    [-0.02, 0.03, 0.01],
    [0.01, -0.02, 0.03],
    [0.02, 0.00, -0.01]
]
mat = np.array(rows)
res_cscv = pbo_cscv(mat, n_blocks=4)

print(f"样本期 T: {res_cscv.n_samples}, 候选策略数 N: {res_cscv.n_strategies}, 块数 S: {res_cscv.n_blocks}")
print(f"总划分组合数 C(4, 2): {res_cscv.n_splits}")
print(f"PBO (CSCV 连续口径 lambda <= 0):     {res_cscv.pbo_cscv:.4f} ({int(res_cscv.pbo_cscv * res_cscv.n_splits)}/6)")
print(f"PBO (定义 2.2 严格口径 r_bar < N/2): {res_cscv.pbo_strict:.4f} (4/6)")
expected_best = [0, 1, 0, 2, 0, 2]
expected_ranks = [1.0, 1.0, 1.0, 2.0, 1.0, 2.0]
expected_omega = [0.25, 0.25, 0.25, 0.50, 0.25, 0.50]
expected_lambdas = [-1.098612, -1.098612, -1.098612, 0.0, -1.098612, 0.0]
np.testing.assert_array_equal([p.best_strategy_index for p in res_cscv.partitions], expected_best)
np.testing.assert_allclose([p.oos_rank_best for p in res_cscv.partitions], expected_ranks, rtol=0.0, atol=1e-12)
np.testing.assert_allclose([p.relative_rank_omega for p in res_cscv.partitions], expected_omega, rtol=0.0, atol=1e-12)
np.testing.assert_allclose([p.logit_lambda for p in res_cscv.partitions], expected_lambdas, rtol=0.0, atol=1e-6)
print("6 split selection/rank/omega/lambda assertions passed")

print("\n全部 6 种划分详细结果:")
for p in res_cscv.partitions:
    print(f"划分 {p.split_index}: IS块={p.is_blocks}, 最优策略={p.best_strategy_index}, "
          f"IS SR={p.is_sharpe_best:.4f}, OOS SR={p.oos_sharpe_best:.4f}, "
          f"OOS秩={p.oos_rank_best:.1f}, omega_bar={p.relative_rank_omega:.2f}, "
          f"lambda={p.logit_lambda:+.4f}, 过拟合={p.is_overfit_cscv}")

## 6. 冠军过拟合实验（The Champion Experiment: M=200 随机策略）

In [ ]:
ch32 = load_code_module("ch32_performance_statistics.py")
results_champ = ch32.run_champion_experiment([20261032])
seed_data = results_champ["seeds_evaluated"][0]

print(f"试验数 M: {results_champ['trials_m']}, 样本天数 T: {results_champ['sample_length_t']}")
print(f"随机挑选出的最优冠军策略年化夏普:  {seed_data['champion_annualized_sharpe']:.4f}")
print(f"朴素单次正态检验 p 值:             {seed_data['unadjusted_p_value']:.6f} (已知零技能模拟；该检验忽略搜索)")
print(f"Bonferroni 调整后 p 值:            {seed_data['bonferroni_adjusted_p']:.4f} (无法拒绝 H0)")
print(f"Holm 调整后 p 值:                  {seed_data['holm_adjusted_p']:.4f}")
print(f"DSR 随机最大门槛 SR0 (年化):       {seed_data['dsr_threshold_sr0_ann']:.4f}")
print(f"紧缩夏普比率 DSR:                  {seed_data['deflated_sharpe_ratio']:.4f} (远低于 0.95)")
print(f"CSCV 回测过拟合概率 PBO:           {seed_data['cscv_pbo']:.2%}")
print(f"样本内向样本外夏普衰减:            {seed_data['cscv_mean_is_sharpe_ann']:.2f} -> {seed_data['cscv_mean_oos_sharpe_ann']:.2f}")
print(f"样本外亏损概率 (Prob Loss):        {seed_data['cscv_prob_loss']:.2%}")

## 7. 历史数据诊断（Kenneth French 10 Industries 1200 个月；逐月扣除 FF3 RF）

In [ ]:
if RUN_FRENCH:
    results_fr = ch32.run_french_market_example()
    
    print(f"数据集:         {results_fr['dataset_name']}")
    print(f"快照日期:       {results_fr['snapshot_date']}")
    print(f"风险基准:       {results_fr['risk_free_benchmark']}")
    print(f"Point-in-time:  {results_fr['is_point_in_time']}")
    print(f"来源限制:       {results_fr['source_limitation']}")
    print(f"样本跨度:       {results_fr['sample_start']} 至 {results_fr['sample_end']} (共 {results_fr['sample_months']} 个月)")
    print(f"子集块数 S:     {results_fr['s_blocks']}, 总划分数 C(12, 6): {results_fr['n_splits']}")
    print(f"全市场 PBO (lambda <= 0): {results_fr['pbo_cscv']:.2%}")
    print(f"严格秩 PBO:              {results_fr['pbo_strict']:.2%}")
    print(f"样本外亏损概率: {results_fr['prob_loss']:.2%}")
    print(f"平均 IS 夏普:   {results_fr['mean_is_sharpe_ann']:.4f}")
    print(f"平均 OOS 夏普:  {results_fr['mean_oos_sharpe_ann']:.4f}")
    print(f"平均衰减:       {results_fr['mean_degradation_ann']:.4f}")
    print(f"衰减回归斜率:   {results_fr['degradation_slope']:.4f}")
    
    df_stats = pd.DataFrame(results_fr["industry_statistics"])
    print("\n10 个行业资产全历史指标:")
    print(df_stats.to_string(index=False))
    print("\n边界：这是当前修订历史，不是 point-in-time 数据；行业组合不含 fills、费用或容量，不能据此推断可交易 alpha 或上线适格性。")
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")
